# Day 9: Snowflake Security, Dynamic Masking & dbt Gold Marts

This notebook simulates Gold metric calculations (30-day readmissions) and role-based column masking using DuckDB and Pandas.

In [ ]:
import pandas as pd
import duckdb

con = duckdb.connect(database=':memory:')
print("DuckDB Initialized for Gold Mart Simulation.")


In [ ]:
# 1. Simulate Gold Metric (30-Day Readmission dbt model)
con.execute("""
    CREATE TABLE condition_occurrence (
        person_source_value VARCHAR,
        condition_concept_id INT,
        condition_start_date DATE
    );
    INSERT INTO condition_occurrence VALUES 
        ('P1', 32020, '2023-01-01'), -- Initial admission
        ('P1', 32020, '2023-01-20'), -- Readmission within 30 days
        ('P2', 32020, '2023-01-05'), -- Initial admission
        ('P2', 32020, '2023-03-10'); -- Readmission > 30 days
""")

gold_mart_query = """
WITH condition_events AS (
    SELECT 
        person_source_value as patient_id,
        condition_start_date as admission_date
    FROM condition_occurrence
    WHERE condition_concept_id = 32020
),
ordered_admissions AS (
    SELECT 
        patient_id,
        admission_date,
        LAG(admission_date) OVER (PARTITION BY patient_id ORDER BY admission_date) as prev_admission_date
    FROM condition_events
)
SELECT 
    patient_id,
    admission_date,
    prev_admission_date,
    DATE_DIFF('day', prev_admission_date, admission_date) as days_since_last_admission,
    CASE 
        WHEN DATE_DIFF('day', prev_admission_date, admission_date) <= 30 THEN 1 
        ELSE 0 
    END as is_30_day_readmission
FROM ordered_admissions
WHERE prev_admission_date IS NOT NULL
"""

print("Gold Mart: 30-Day Readmissions Results")
print(con.execute(gold_mart_query).fetchdf())


In [ ]:
# 2. Simulate Dynamic Data Masking
raw_patients = pd.DataFrame({
    'patient_id': ['P1', 'P2'],
    'ssn': ['123-45-6789', '987-65-4321'],
    'dob': ['1980-05-15', '1992-11-20']
})

def query_patients(role):
    df = raw_patients.copy()
    if role == 'DATA_ANALYST':
        df['ssn'] = '***-**-****'
        df['dob'] = pd.to_datetime(df['dob']).dt.year.astype(str) + '-01-01' # Truncate to year
    elif role == 'CLINICIAN':
        pass # Full access
    else:
        df['ssn'] = '***-**-****'
        df['dob'] = 'REDACTED'
    return df

print("\n--- Clinician View ---")
print(query_patients('CLINICIAN'))

print("\n--- Data Analyst View ---")
print(query_patients('DATA_ANALYST'))
